<a href="https://colab.research.google.com/github/shreyasat27/mahework2025/blob/main/Paper_hamiltonian(alr_values).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qutip import *

# --- PARAMETERS ---
mu_B = 12.0       # Bohr magneton (arb. units)
B = 10.0          # Magnetic field magnitude (along z)
g1, g2 = 2, 2  # Electron g-factors
J = 0.05         # Exchange coupling constant
A1 = [0.1, 0.1, 0.2]  # Hyperfine for nucleus 1
A2 = [0.5, 0.5, 0.1]  # Hyperfine for nucleus 2
k_S = 0.1        # Singlet recombination rate
omega_drive = 3.3
# --- DEFINE SPIN OPERATORS ---
# Hilbert space order: electron1, electron2, nucleus1, nucleus2

sx_e1 = tensor(sigmax(), qeye(2), qeye(2), qeye(2))
sy_e1 = tensor(sigmay(), qeye(2), qeye(2), qeye(2))
sz_e1 = tensor(sigmaz(), qeye(2), qeye(2), qeye(2))

sx_e2 = tensor(qeye(2), sigmax(), qeye(2), qeye(2))
sy_e2 = tensor(qeye(2), sigmay(), qeye(2), qeye(2))
sz_e2 = tensor(qeye(2), sigmaz(), qeye(2), qeye(2))

sx_n1 = tensor(qeye(2), qeye(2), sigmax(), qeye(2))
sy_n1 = tensor(qeye(2), qeye(2), sigmay(), qeye(2))
sz_n1 = tensor(qeye(2), qeye(2), sigmaz(), qeye(2))

sx_n2 = tensor(qeye(2), qeye(2), qeye(2), sigmax())
sy_n2 = tensor(qeye(2), qeye(2), qeye(2), sigmay())
sz_n2 = tensor(qeye(2), qeye(2), qeye(2), sigmaz())

# --- HAMILTONIAN ---

B_vec = np.array([0, 0, B])  # magnetic field along z

H_Z = mu_B * (
    g1 * (B_vec[0]*sx_e1 + B_vec[1]*sy_e1 + B_vec[2]*sz_e1) +
    g2 * (B_vec[0]*sx_e2 + B_vec[1]*sy_e2 + B_vec[2]*sz_e2)
)

H_HF = (
    A1[0] * sx_e1 * sx_n1 + A1[1] * sy_e1 * sy_n1 + A1[2] * sz_e1 * sz_n1 +
    A2[0] * sx_e2 * sx_n2 + A2[1] * sy_e2 * sy_n2 + A2[2] * sz_e2 * sz_n2
)

H_J = J * (sx_e1 * sx_e2 + sy_e1 * sy_e2 + sz_e1 * sz_e2)

H = [H_Z + H_HF + H_J,
     [mu_B * (sx_e1 + sx_e2), 'cos(w*t)']]

args = {'w': omega_drive}


# --- INITIAL STATE ---

# Electron singlet state |S> = (|01> - |10>)/sqrt(2)
singlet_elec = (tensor(basis(2,0), basis(2,1)) - tensor(basis(2,1), basis(2,0))).unit()

# Nuclear spins in mixed state (identity / 4)
rho_nuc = tensor(qeye(2)/2, qeye(2)/2)

# Full initial state: singlet electrons tensor nuclear mixed state
rho0 = tensor(ket2dm(singlet_elec), rho_nuc)

# --- COLLAPSE OPERATORS ---

# Singlet projector on electron spins
P_S_elec = ket2dm(singlet_elec)

# Lift singlet projector to full space by tensor nuclear identity
P_S_full = tensor(P_S_elec, qeye(2), qeye(2))

# Collapse operator for singlet recombination
c_ops = [np.sqrt(k_S) * P_S_full]

# --- OBSERVABLES ---

# Triplet projector = Identity on electrons - singlet projector
P_T_elec = tensor(qeye(2), qeye(2)) - P_S_elec
P_T_full = tensor(P_T_elec, qeye(2), qeye(2))

observables = [P_S_full, P_T_full]

# --- TIME EVOLUTION ---

tlist = np.linspace(0, 60, 500)
result = mesolve(H, rho0, tlist, c_ops, observables, args=args)
#result = mesolve(H, rho0, tlist, c_ops,observables)

# --- PLOTTING ---

plt.plot(tlist, result.expect[0], label='Singlet population')
plt.plot(tlist, result.expect[1], label='Triplet population')
plt.xlabel('Time')
plt.ylabel('Population')
plt.legend()
plt.show()